# Inference (ONNX)

This notebook covers the complete inference pipeline for the trained Mask R-CNN v1
model — from exporting the PyTorch checkpoint to ONNX format through to validated
deployment-ready inference.

The workflow:

- **Model setup:** reconstruct the small-object Mask R-CNN v1 architecture with
  custom FPN anchor scales and the official map_17 class taxonomy;
- **ONNX export:** wrap the model for single-image inference with a fixed output
  layout (padded to `max_detections=100`), export with `opset_version=12`, and
  validate the graph with the ONNX checker;
- **Runtime inference:** load the exported model with ONNX Runtime (GPU or CPU),
  preprocess images to the fixed export resolution, and return boxes, scores,
  labels, and instance masks;
- **Validation:** compare PyTorch and ONNX outputs image-by-image using IoU-based
  detection matching to confirm numerical equivalence before deployment.

The inference image uses only `onnxruntime-gpu`, `Pillow`, `numpy`, and
`opencv-python-headless` — no PyTorch or torchvision dependency at runtime.

## Check hardware

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [1]:
!nvidia-smi

Wed Jul  1 16:08:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install Packages

In [2]:
!pip install torchmetrics pycocotools gdown tqdm tensorboard albumentations onnx "onnxruntime-gpu==1.20.1" onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 129.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.2 MB/s eta 0:00:00


In [ ]:
!curl https://rclone.org/install.sh | sudo bash

## Import Packages

In [3]:
import os
import pandas as pd
import json
from tqdm import tqdm

import ast
from PIL import Image, ImageDraw
import os
import torch
import math

import numpy as np
from torchvision.io import read_image
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F
from torch.utils.data import Dataset, Subset, DataLoader, random_split, WeightedRandomSampler
from torch import nn
import random
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torch.utils.tensorboard import SummaryWriter
from matplotlib import colors, pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

from torchvision.ops import box_iou

from torchmetrics.detection import (
    MeanAveragePrecision,
    IntersectionOverUnion,
    GeneralizedIntersectionOverUnion,
    DistanceIntersectionOverUnion,
    CompleteIntersectionOverUnion,
)
from torchmetrics.segmentation import DiceScore, GeneralizedDiceScore, MeanIoU

%matplotlib inline

## Set Environments Path

In [4]:
def is_google_colab() -> bool:
    """Check whether the notebook is running in Google Colab.
    
    Returns:
        bool: ``True`` in a Colab runtime; otherwise ``False``.
    """
    try:
        import google.colab
        return True
    except ImportError:
        return False


IN_COLAB = is_google_colab()

if IN_COLAB:
    ROOT_PATH = "/content"
    DATAFRAME_PATH = "/content/drive/MyDrive/taco_trash"
    DEFAULT_RUNS_PATH = "/content/drive/MyDrive/taco_trash"
else:
    ROOT_PATH = "/home/ubuntu"
    DATAFRAME_PATH = os.path.join(ROOT_PATH, "taco_trash")
    DEFAULT_RUNS_PATH = os.path.join(ROOT_PATH, "taco_trash")

TACO_CLASSIFICATION_PATH = os.path.join(ROOT_PATH, "taco")

In [5]:
if is_google_colab():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    !curl https://rclone.org/install.sh | sudo bash

Mounted at /content/drive


In [6]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [7]:
torch.cuda.current_device()
torch.cuda.get_device_name(0)
device = torch.device("cuda")
device

device(type='cuda')

## Download Data

In [8]:
!gdown 1CUxA2TcZFhVJWvNH4qwlOSGDPlxodRPF
!unzip taco.zip -d $ROOT_PATH

Downloading...
From (original): https://drive.google.com/uc?id=1CUxA2TcZFhVJWvNH4qwlOSGDPlxodRPF
From (redirected): https://drive.google.com/uc?id=1CUxA2TcZFhVJWvNH4qwlOSGDPlxodRPF&confirm=t&uuid=723b125a-ecb0-483b-afd2-8fc6b3f829f4
To: /content/taco.zip
100% 5.23G/5.23G [01:30<00:00, 57.7MB/s]
Archive:  taco.zip
   creating: /content/content/drive/MyDrive/taco_dataset/taco/
  inflating: /content/content/drive/MyDrive/taco_dataset/taco/annotations.json  
   creating: /content/content/drive/MyDrive/taco_dataset/taco/batch_14/
  inflating: /content/content/drive/MyDrive/taco_dataset/taco/batch_14/000011.jpg  
  inflating: /content/content/drive/MyDrive/taco_dataset/taco/batch_14/000007.jpg  
  inflating: /content/content/drive/MyDrive/taco_dataset/taco/batch_14/000004.jpg  
  inflating: /content/content/drive/MyDrive/taco_dataset/taco/batch_14/000000.jpg  
  inflating: /content/content/drive/MyDrive/taco_dataset/taco/batch_14/000014.jpg  
  inflating: /content/content/drive/MyDrive/taco_

In [9]:
!gdown 1lj6H8pIbHjgrgNRWHoc41Nr8bi0FZ7mt # annotations_processed.csv

Downloading...
From: https://drive.google.com/uc?id=1lj6H8pIbHjgrgNRWHoc41Nr8bi0FZ7mt
To: /content/annotations_processed.csv
100% 3.10M/3.10M [00:00<00:00, 188MB/s]


In [10]:
def parse_bbox(v):
    if isinstance(v, str):
        return ast.literal_eval(v)
    return v
    
if is_google_colab():
    df = pd.read_csv(os.path.join(ROOT_PATH, "annotations_processed.csv"))
else:
    ann_path = os.path.join(TACO_CLASSIFICATION_PATH, "annotations.json")
    with open(ann_path, "r") as f:
        data = json.load(f)

    print(data.keys())

    for key in data.keys():
        if isinstance(data[key], list):
            print(key, len(data[key]))

    df_images = pd.DataFrame(data["images"])
    df_annotations = pd.DataFrame(data["annotations"])
    df_categories = pd.DataFrame(data["categories"])
    df_scene_annotations = pd.DataFrame(data["scene_annotations"])
    df_scene_categories = pd.DataFrame(data["scene_categories"])

    df = df_annotations.merge(
        df_images,
        left_on="image_id",
        right_on="id",
        how="left",
        suffixes=("", "_img"),
    ).merge(
        df_categories,
        left_on="category_id",
        right_on="id",
        how="left",
        suffixes=("", "_cat"),
    ).rename(
        columns={
            "id_x": "annotation_id",
            "id_y": "category_id_full",
            "name": "category_name",
        }
    )
    df['file_path'] = df['file_name'].map(lambda x: os.path.join(TACO_CLASSIFICATION_PATH, x) )

## Model functions

In [11]:
from torchvision.models.detection.anchor_utils import AnchorGenerator

def get_model_small_objects(num_classes, train_mode="heads"):
    """Create Mask R-CNN v1 with small-object anchor scales.
    
    Args:
        num_classes (int): Total class count including the background class.
        train_mode (str): Trainable component set: ``predictors``, ``heads``, or ``all``.
    
    Returns:
        torch.nn.Module: Configured model moved to the active device.
    """
    anchor_generator = AnchorGenerator(
        # Default was ((32,),(64,),(128,),(256,),(512,)) — one size per FPN level
        # P2 now gets small anchors (8,16,32) to catch tiny objects
        sizes=(
            (8,  16,  32),    # P2 — small anchors for tiny objects
            (32,  64, 128),   # P3
            (64, 128, 256),   # P4
            (128, 256, 512),  # P5
            (256, 512, 1024), # P6
        ),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5,  # 3 ratios × 3 sizes = 9 anchors per location
    )

    model = torchvision.models.detection.maskrcnn_resnet50_fpn(
        weights=None,
        min_size=(800, 1024, 1280),  # randomly sampled per image during training
        max_size=1536,
        rpn_anchor_generator=anchor_generator,
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask,
        256,
        num_classes,
    )

    for p in model.parameters():
        p.requires_grad = False

    if train_mode == "predictors":
        for p in model.roi_heads.box_predictor.parameters():
            p.requires_grad = True
        for p in model.roi_heads.mask_predictor.parameters():
            p.requires_grad = True

    elif train_mode == "heads":
        for p in model.rpn.parameters():
            p.requires_grad = True
        for p in model.roi_heads.parameters():
            p.requires_grad = True

    elif train_mode == "all":
        for p in model.parameters():
            p.requires_grad = True

    else:
        raise ValueError(f"Unknown train_mode: {train_mode}")

    model.to(device)
    return model

In [12]:
def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None, device="cuda"):
    """Restore model and optional training state from disk.
    
    Args:
        path (str): Checkpoint file path.
        model (torch.nn.Module): Torchvision detection model.
        optimizer (torch.optim.Optimizer): Optimizer for trainable model parameters.
        scheduler (Any | None): Optional learning-rate scheduler.
        scaler (torch.amp.GradScaler | None): Optional mixed-precision gradient scaler.
        device (torch.device | str): Device used for model computation.
    
    Returns:
        dict: Loaded checkpoint dictionary.
    """
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)

    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if scaler is not None and "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    return checkpoint

In [13]:
MAP_17_RAW = """Aerosol,Can
Aluminium foil,Aluminium foil
Battery,Other
Aluminium blister pack,Other
Carded blister pack,Other
Clear plastic bottle,Plastic bottle
Glass bottle,Glass bottle
Other plastic bottle,Plastic bottle
Plastic bottle cap,Plastic bottle cap
Metal bottle cap,Metal bottle cap
Broken glass,Other
Drink can,Can
Food Can,Can
Corrugated carton,Carton
Drink carton,Carton
Egg carton,Carton
Meal carton,Carton
Other carton,Carton
Paper cup,Cup
Disposable plastic cup,Cup
Foam cup,Cup
Glass cup,Cup
Other plastic cup,Cup
Food waste,Other
Plastic lid,Plastic lid
Metal lid,Other
Magazine paper,Paper
Tissues,Paper
Wrapping paper,Paper
Normal paper,Paper
Paper bag,Paper
Plastified paper bag,Paper
Pizza box,Carton
Garbage bag,Plastic film
Single-use carrier bag,Plastic film
Polypropylene bag,Plastic film
Produce bag,Plastic film
Cereal bag,Plastic film
Bread bag,Plastic film
Plastic film,Plastic film
Crisp packet,Wrapper
Other plastic wrapper,Wrapper
Retort pouch,Wrapper
Spread tub,Plastic container
Tupperware,Plastic container
Disposable food container,Plastic container
Foam food container,Plastic container
Other plastic container,Plastic container
Plastic glooves,Other
Plastic utensils,Other
Pop tab,Pop tab
Rope & strings,Other
Scrap metal,Other
Shoe,Other
Six pack rings,Plastic film
Squeezable tube,Other
Plastic straw,Straw
Paper straw,Straw
Styrofoam piece,Styrofoam piece
Toilet tube,Carton
Unlabeled litter,Other
Glass jar,Other
Other plastic,Other
Cigarette,Other"""

MAP_17 = dict(line.split(",", 1)for line in MAP_17_RAW.strip().splitlines())


def make_map17_taxonomy_df(df):
    """Map fine-grained categories into the project map-17 taxonomy.
    
    Args:
        df (pd.DataFrame): Object-level annotation dataframe.
    
    Returns:
        tuple[pd.DataFrame, dict, int, dict]: Relabeled annotations and class mappings.
    """
    df_new = df.copy()

    df_new["label_map17"] = df_new["category_name"].map(MAP_17).fillna("Other")

    # Print what's in the dataset vs what map_17 covers
    unmapped = set(df_new["category_name"].unique()) - set(MAP_17.keys())
    if unmapped:
        print(f"Unmapped category_names (→ Other): {unmapped}")

    class_counts = df_new["label_map17"].value_counts()
    print(f"\nClass distribution ({len(class_counts)} classes):")
    print(class_counts.to_string())

    label_values = sorted(df_new["label_map17"].unique())
    label_map    = {name: i + 1 for i, name in enumerate(label_values)}
    num_classes  = len(label_map) + 1
    id_to_name   = {v: k for k, v in label_map.items()}

    return df_new, label_map, num_classes, id_to_name

In [14]:
df_map17, label_map_map17, num_classes_map17, id_to_name_map17 = make_map17_taxonomy_df(df)


Class distribution (17 classes):
label_map17
Other                 1732
Plastic film           551
Plastic bottle         335
Wrapper                299
Can                    273
Carton                 251
Plastic bottle cap     209
Cup                    192
Paper                  175
Straw                  161
Styrofoam piece        112
Glass bottle           104
Pop tab                 99
Metal bottle cap        80
Plastic lid             77
Plastic container       72
Aluminium foil          62


## Convert Mask R-CNN to ONNX

The main challenge: Mask R-CNN expects `List[Tensor]` as input and returns `List[Dict]` as output — neither works natively with ONNX. The solution is a wrapper that bridges the gap.

In [ ]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np


class MaskRCNNONNXWrapper(nn.Module):
    """Wraps Mask R-CNN for ONNX export.

    Converts the model's native List[Tensor] → List[Dict] interface into a
    single-image tensor → fixed-layout tensor interface that ONNX can trace.
    Predictions are padded or truncated to ``max_detections`` so the output
    shape is static. Masks are resized to ``mask_size`` to prevent OOM on
    high-resolution inputs.

    Args:
        model (nn.Module): Configured Mask R-CNN model in eval mode.
        max_detections (int): Maximum number of detections in the output.
            Outputs are zero-padded up to this count. Defaults to 100.
        mask_size (tuple[int, int]): Fixed (H, W) to which all output masks
            are resized. Should match the export dummy input size.
            Defaults to ``(1024, 1280)``.
    """

    def __init__(self, model, max_detections=100, mask_size=(1024, 1280)):
        super().__init__()
        self.model = model
        self.max_detections = max_detections
        self.mask_h, self.mask_w = mask_size   # fixed output resolution

    def forward(self, image):
        outputs = self.model([image])
        out = outputs[0]

        n   = out["boxes"].shape[0]
        pad = self.max_detections - n

        boxes  = torch.cat([out["boxes"],
                             torch.zeros(pad, 4, device=image.device)])
        scores = torch.cat([out["scores"],
                             torch.zeros(pad, device=image.device)])
        labels = torch.cat([out["labels"],
                             torch.zeros(pad, device=image.device, dtype=torch.int64)])

        # resize masks to fixed size to avoid OOM on large input images
        raw_masks = out["masks"].float()              # [N, 1, H_orig, W_orig]
        if raw_masks.shape[2] != self.mask_h or raw_masks.shape[3] != self.mask_w:
            raw_masks = torch.nn.functional.interpolate(
                raw_masks, size=(self.mask_h, self.mask_w),
                mode="bilinear", align_corners=False,
            )
        masks = torch.cat([raw_masks.squeeze(1),
                           torch.zeros(pad, self.mask_h, self.mask_w, device=image.device)])

        return boxes, scores, labels, masks, torch.tensor(n)

In [ ]:
def export_to_onnx(checkpoint_path, num_classes, output_path="mask_rcnn.onnx"):
    """Export a trained Mask R-CNN checkpoint to ONNX format.

    Loads the checkpoint, fixes inference-time model settings (single input
    scale, score threshold, NMS threshold), wraps the model with
    ``MaskRCNNONNXWrapper``, traces with a dummy input, and validates the
    exported graph with the ONNX checker.

    Args:
        checkpoint_path (str): Path to the ``.pt`` checkpoint file produced
            by the training pipeline.
        num_classes (int): Total number of classes including background,
            must match the checkpoint's predictor head shape.
        output_path (str): Destination path for the exported ``.onnx`` file.
            Defaults to ``"mask_rcnn.onnx"``.

    Returns:
        str: The resolved ``output_path`` of the saved ONNX model.
    """

    # ── Load model ────────────────────────────────────────────────────────────
    model = get_model_small_objects(num_classes=num_classes, train_mode="all")
    load_checkpoint(checkpoint_path, model, device=torch.device("cpu"))
    model.eval()

    # Fix inference settings before export
    model.transform.min_size         = (1024,)   # no random multi-scale at inference
    model.transform.max_size         = 1536
    model.roi_heads.score_thresh     = 0.05
    model.roi_heads.detections_per_img = 100
    model.roi_heads.nms_thresh       = 0.5

    wrapper = MaskRCNNONNXWrapper(model, max_detections=100)
    wrapper.eval()

    # ── Dummy input ───────────────────────────────────────────────────────────
    dummy = torch.zeros(3, 1024, 1280)   # [C, H, W] — no batch dim

    # ── Export ────────────────────────────────────────────────────────────────
    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            dummy,
            output_path,
            opset_version=12,
            dynamo=False,
            input_names=["image"],
            output_names=["boxes", "scores", "labels", "masks", "num_detections"],
            dynamic_axes={
                "image":  {1: "height", 2: "width"},   # variable spatial size
                "boxes":  {0: "detections"},
                "scores": {0: "detections"},
                "labels": {0: "detections"},
                "masks":  {0: "detections"},
            },
        )

    # ── Validate ──────────────────────────────────────────────────────────────
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    print(f"ONNX model saved and validated: {output_path}")
    return output_path

## Export to ONNX

In [17]:
export_to_onnx(
    checkpoint_path=os.path.join(DATAFRAME_PATH, "checkpoints", "v1_small_objects_cosine_stage_3_25_best_model.pt"),
    num_classes=num_classes_map17,
    output_path=os.path.join(DATAFRAME_PATH, "checkpoints", "mask_rcnn_v1.onnx"),
)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 245MB/s]
/tmp/ipykernel_1844/3815171073.py:22: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:4790: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  * torch.tensor(scale_factors[i], dtype=torch.float32)
/usr/local/lib/python3.12/dist-packages/torchvision/ops/boxes.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.d

ONNX model saved and validated: /content/drive/MyDrive/taco_trash/checkpoints/mask_rcnn_v1.onnx


'/content/drive/MyDrive/taco_trash/checkpoints/mask_rcnn_v1.onnx'

## Inference with ONNX Runtime

In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image


def load_onnx_session(model_path, use_gpu=True):
    """Create an ONNX Runtime inference session.

    Args:
        model_path (str): Path to the exported ``.onnx`` model file.
        use_gpu (bool): If ``True``, attempts to use ``CUDAExecutionProvider``
            before falling back to CPU. Defaults to ``True``.

    Returns:
        onnxruntime.InferenceSession: Ready-to-use inference session.
    """

    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if use_gpu else
        ["CPUExecutionProvider"]
    )
    return ort.InferenceSession(model_path, providers=providers)


def preprocess(image_path, size=(1024, 1280)):
    """Load and resize an image to the fixed ONNX export resolution.

    Opens the image, resizes to ``(height, width)`` without preserving aspect
    ratio, normalises pixel values to ``[0, 1]``, and returns a CHW float32
    array ready to pass to the ONNX session.

    Args:
        image_path (str): Path to the input image file.
        size (tuple[int, int]): Target ``(height, width)`` in pixels. Must
            match the dummy input size used during ONNX export.
            Defaults to ``(1024, 1280)``.

    Returns:
        numpy.ndarray: Float32 array of shape ``[3, H, W]`` with values in
        ``[0.0, 1.0]``.
    """

    img = Image.open(image_path).convert("RGB")
    img = img.resize((size[1], size[0]))
    arr = np.array(img, dtype=np.float32) / 255.0   # [H, W, 3]
    arr = arr.transpose(2, 0, 1)                     # [3, H, W]
    return arr


def predict(session, image_path, score_thresh=0.20):
    """Run inference on a single image using the ONNX Runtime session.

    Preprocesses the image, runs the ONNX model, and returns only the
    detections that were populated by the model (up to ``max_detections``).

    Args:
        session (onnxruntime.InferenceSession): Loaded ONNX Runtime session.
        image_path (str): Path to the input image file.
        score_thresh (float): Minimum confidence score. Detections below this
            threshold were already filtered during export; this value should
            match ``model.roi_heads.score_thresh`` used at export time.
            Defaults to ``0.20``.

    Returns:
        dict: Dictionary with keys:
            - ``boxes``  (numpy.ndarray): ``[N, 4]`` xyxy bounding boxes.
            - ``scores`` (numpy.ndarray): ``[N]`` confidence scores.
            - ``labels`` (numpy.ndarray): ``[N]`` integer class IDs.
            - ``masks``  (numpy.ndarray): ``[N, H, W]`` float instance masks.
    """

    image = preprocess(image_path)                   # [3, H, W]

    boxes, scores, labels, masks, n_det = session.run(
        None, {"image": image}
    )

    n = int(n_det[0])
    return {
        "boxes":  boxes[:n],    # [N, 4]  xyxy
        "scores": scores[:n],   # [N]
        "labels": labels[:n],   # [N]
        "masks":  masks[:n],    # [N, H, W]
    }

## Helper functions

In [ ]:
import numpy as np
import torch
from PIL import Image
import onnxruntime as ort

from PIL import Image, ExifTags
import torch
import numpy as np

def load_image_with_exif(file_path):
    """Load image and correct EXIF orientation. Replaces read_image()."""
    image = Image.open(file_path).convert("RGB")
    try:
        exif = image._getexif()
        if exif:
            for tag, value in exif.items():
                if ExifTags.TAGS.get(tag) == "Orientation":
                    if value == 3:
                        image = image.rotate(180, expand=True)
                    elif value == 6:
                        image = image.rotate(270, expand=True)
                    elif value == 8:
                        image = image.rotate(90, expand=True)
                    break
    except Exception:
        pass
    return torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0

# ── Helpers ───────────────────────────────────────────────────────────────────

def run_pytorch(model, image_path, score_thresh=0.05, detections_per_img=100):
    """Run the PyTorch Mask R-CNN model on a single preprocessed image.

    Applies the same fixed-size preprocessing as ``run_onnx`` so that both
    models receive identical pixel data, enabling a fair numerical comparison.

    Args:
        model (nn.Module): Loaded Mask R-CNN model in eval mode.
        image_path (str): Path to the input image file.
        score_thresh (float): Confidence threshold passed to
            ``model.roi_heads.score_thresh``. Defaults to ``0.05``.
        detections_per_img (int): Maximum detections returned by the model.
            Should match ``max_detections`` used during ONNX export.
            Defaults to ``100``.

    Returns:
        dict: Dictionary with keys ``boxes``, ``scores``, ``labels``,
        ``masks`` as numpy arrays.
    """

    arr = preprocess(image_path, size=(1024, 1280))   # ← same resize as ONNX
    img = torch.from_numpy(arr)                        # [3, 1024, 1280]
    model.eval()
    model.transform.min_size         = (1024,)
    model.roi_heads.score_thresh     = score_thresh
    model.roi_heads.detections_per_img = detections_per_img

    with torch.no_grad():
        outputs = model([img.to(next(model.parameters()).device)])

    out = outputs[0]
    return {
        "boxes":  out["boxes"].cpu().numpy(),
        "scores": out["scores"].cpu().numpy(),
        "labels": out["labels"].cpu().numpy(),
        "masks":  out["masks"].squeeze(1).cpu().numpy(),
    }

def run_onnx(session, image_path, score_thresh=0.05):
    """Run the exported ONNX model on a single image.

    Preprocesses the image to the fixed export resolution ``(1024, 1280)``
    and returns only the valid (non-padded) detections as reported by the
    ``num_detections`` output.

    Args:
        session (onnxruntime.InferenceSession): Loaded ONNX Runtime session.
        image_path (str): Path to the input image file.
        score_thresh (float): Unused at inference time (filtering was baked
            in at export); kept for API symmetry with ``run_pytorch``.
            Defaults to ``0.05``.

    Returns:
        dict: Dictionary with keys ``boxes``, ``scores``, ``labels``,
        ``masks`` as numpy arrays containing only valid detections.
    """

    img = preprocess(image_path, size=(1024, 1280))   # resize to ONNX export size
    boxes, scores, labels, masks, n_det = session.run(
        None, {"image": img}
    )
    n = int(n_det)
    return {
        "boxes":  boxes[:n],
        "scores": scores[:n],
        "labels": labels[:n],
        "masks":  masks[:n],
    }


# ── Per-image comparison ──────────────────────────────────────────────────────

def compare_outputs(pt_out, onnx_out, image_path, rtol=1e-3, atol=1e-3, match_iou=0.5):
    """Compare PyTorch and ONNX model outputs for one image.

    Uses IoU-based greedy matching to pair detections across the two models,
    avoiding false failures caused by different NMS ordering in dense scenes.
    Checks score agreement, label equality, and mask IoU on matched pairs.

    Args:
        pt_out (dict): Output from ``run_pytorch`` with keys ``boxes``,
            ``scores``, ``labels``, ``masks``.
        onnx_out (dict): Output from ``run_onnx`` with the same keys.
        image_path (str): Image path used only for display in the result dict.
        rtol (float): Relative tolerance for score comparison. Defaults to
            ``1e-3``.
        atol (float): Absolute tolerance for score comparison. Defaults to
            ``1e-3``.
        match_iou (float): Minimum box IoU to consider two detections the
            same object. Defaults to ``0.5``.

    Returns:
        dict: Result dictionary containing:
            - ``passed`` (bool): ``True`` if all checks passed.
            - ``issues`` (list[str]): Descriptions of any failures.
            - ``n_pytorch``, ``n_onnx``, ``n_matched``, ``n_unmatched`` (int).
            - ``max_score_diff`` (float): Max absolute score diff on matched pairs.
            - ``mean_mask_iou`` (float): Mean mask IoU on matched pairs.
    """

    from torchvision.ops import box_iou
    results = {"image": image_path, "passed": True, "issues": []}

    n_pt   = len(pt_out["scores"])
    n_onnx = len(onnx_out["scores"])

    if n_pt == 0 and n_onnx == 0:
        results["issues"].append("no detections in either model")
        return results

    # ── Match each PyTorch detection to best ONNX detection by IoU ────────
    pt_boxes   = torch.from_numpy(pt_out["boxes"].astype(np.float32))
    onnx_boxes = torch.from_numpy(onnx_out["boxes"].astype(np.float32))

    if n_pt == 0 or n_onnx == 0:
        results["issues"].append(
            f"one model has no detections: PyTorch={n_pt}, ONNX={n_onnx}"
        )
        results["passed"] = False
        return results

    iou_matrix = box_iou(pt_boxes, onnx_boxes).numpy()   # [n_pt, n_onnx]

    matched_pt, matched_onnx = [], []
    used_onnx = set()

    for i in range(n_pt):
        best_j = int(np.argmax(iou_matrix[i]))
        if iou_matrix[i, best_j] >= match_iou and best_j not in used_onnx:
            matched_pt.append(i)
            matched_onnx.append(best_j)
            used_onnx.add(best_j)

    n_matched   = len(matched_pt)
    n_unmatched = n_pt - n_matched

    if n_unmatched > 0:
        results["issues"].append(
            f"{n_unmatched}/{n_pt} PyTorch detections unmatched in ONNX (IoU<{match_iou})"
        )
        if n_unmatched / n_pt > 0.15:     # fail only if >15% unmatched
            results["passed"] = False

    if n_matched == 0:
        results["issues"].append("no detections could be matched")
        results["passed"] = False
        return results

    pt_s = pt_out["scores"][matched_pt]
    pt_l = pt_out["labels"][matched_pt]
    pt_m = pt_out["masks"][matched_pt]

    onnx_s = onnx_out["scores"][matched_onnx]
    onnx_l = onnx_out["labels"][matched_onnx]
    onnx_m = onnx_out["masks"][matched_onnx]

    # 1. Scores
    score_diff = np.abs(pt_s - onnx_s)
    if not np.allclose(pt_s, onnx_s, rtol=rtol, atol=atol):
        results["issues"].append(
            f"scores differ on matched: max={score_diff.max():.6f} mean={score_diff.mean():.6f}"
        )
        results["passed"] = False

    # 2. Labels
    label_mismatch = (pt_l != onnx_l).sum()
    if label_mismatch > 0:
        results["issues"].append(f"labels differ: {label_mismatch}/{n_matched} matched")
        results["passed"] = False

    # 3. Masks
    mask_ious = []
    for pt_m_i, onnx_m_i in zip(pt_m, onnx_m):
        pt_bin   = (pt_m_i   > 0.5).astype(np.float32)
        onnx_bin = (onnx_m_i > 0.5).astype(np.float32)
        inter = (pt_bin * onnx_bin).sum()
        union = (pt_bin + onnx_bin).clip(0, 1).sum()
        mask_ious.append(inter / union if union > 0 else 1.0)

    mean_mask_iou = float(np.mean(mask_ious))
    if mean_mask_iou < 0.95:
        results["issues"].append(f"mask IoU low: mean={mean_mask_iou:.4f}")
        results["passed"] = False

    results.update({
        "n_pytorch":      n_pt,
        "n_onnx":         n_onnx,
        "n_matched":      n_matched,
        "n_unmatched":    n_unmatched,
        "max_score_diff": float(score_diff.max()),
        "mean_mask_iou":  mean_mask_iou,
    })

    return results

In [ ]:
def validate_onnx_export(
    model,
    onnx_path,
    image_paths,
    score_thresh=0.05,
    use_gpu=True,
):
    """Validate an exported ONNX model against its PyTorch source on a set of images.

    For each image, runs both models and calls ``compare_outputs`` to check
    numerical equivalence. Prints a per-image pass/fail summary and aggregate
    statistics across passing images.

    Args:
        model (nn.Module): The original PyTorch Mask R-CNN model used for export.
        onnx_path (str): Path to the exported ``.onnx`` file.
        image_paths (list[str]): Paths to test images. Use a diverse set
            covering different scene densities, object sizes, and lighting
            conditions.
        score_thresh (float): Confidence threshold applied to both models.
            Defaults to ``0.05``.
        use_gpu (bool): Whether to run ONNX Runtime on GPU. Defaults to ``True``.

    Returns:
        list[dict]: One result dict per image as returned by ``compare_outputs``,
        including pass/fail status and numeric diagnostics.
    """

    session = load_onnx_session(onnx_path, use_gpu=use_gpu)

    all_results = []
    passed = 0

    for path in image_paths:
        try:
            pt_out   = run_pytorch(model, path, score_thresh)
            onnx_out = run_onnx(session, path, score_thresh)
            result   = compare_outputs(pt_out, onnx_out, path)
        except Exception as e:
            result = {"image": path, "passed": False, "issues": [str(e)]}

        all_results.append(result)
        status = "✓" if result["passed"] else "✗"
        issues = " | ".join(result.get("issues", []))
        print(f"{status} {path}")
        if issues:
            print(f"    {issues}")

        if result["passed"]:
            passed += 1

    print(f"\n{passed}/{len(image_paths)} images passed validation")

    # Aggregate numeric stats across all passing images
    passing = [r for r in all_results if r["passed"]]
    if passing:
        print(f"max score diff : {max(r['max_score_diff'] for r in passing if r.get('max_score_diff')):.6f}")
        # remove the max_box_diff line
        print(f"mean mask IoU  : {np.mean([r['mean_mask_iou'] for r in passing]):.4f}")

    return all_results

In [44]:
!ls -la /content/taco/batch_10
!pwd

total 159484
drwx------  2 root root    4096 Jan  9 08:16 .
drwx------ 17 root root    4096 Jan  9 08:16 ..
-rw-------  1 root root 2406037 Dec 26  2025 000000.jpg
-rw-------  1 root root 2580129 Dec 26  2025 000001.jpg
-rw-------  1 root root 2454045 Dec 26  2025 000002.jpg
-rw-------  1 root root 2239649 Dec 26  2025 000003.jpg
-rw-------  1 root root 2190916 Dec 26  2025 000004.jpg
-rw-------  1 root root 2480640 Dec 26  2025 000005.jpg
-rw-------  1 root root 1856079 Dec 26  2025 000006.jpg
-rw-------  1 root root 2194320 Dec 26  2025 000007.jpg
-rw-------  1 root root 2710006 Dec 26  2025 000008.jpg
-rw-------  1 root root 2209700 Dec 26  2025 000009.jpg
-rw-------  1 root root 1457262 Dec 26  2025 000010.jpg
-rw-------  1 root root  783321 Dec 26  2025 000011.jpg
-rw-------  1 root root 1659449 Dec 26  2025 000012.jpg
-rw-------  1 root root 2282379 Dec 26  2025 000013.jpg
-rw-------  1 root root 1402882 Dec 26  2025 000014.jpg
-rw-------  1 root root 1872936 Dec 26  2025 000015.

## Run validation across multiple images

In [43]:
# ── Usage ─────────────────────────────────────────────────────────────────────

# Pick a diverse set — mix of sizes, class types, crowded/sparse scenes
TEST_IMAGES = [
    os.path.join(TACO_CLASSIFICATION_PATH, "batch_1", "000001.jpg"),
    os.path.join(TACO_CLASSIFICATION_PATH, "batch_3", "IMG_4980.JPG"),
    os.path.join(TACO_CLASSIFICATION_PATH, "batch_7", "000021.JPG"),
    os.path.join(TACO_CLASSIFICATION_PATH, "batch_10", "000008.jpg"),
    os.path.join(TACO_CLASSIFICATION_PATH, "batch_14", "000006.jpg"),
    # or sample directly from val_loader_map17
]

# Load PyTorch model (already in memory from training)
model_pt = get_model_small_objects(num_classes=num_classes_map17, train_mode="all")
load_checkpoint(
    os.path.join(DATAFRAME_PATH, "checkpoints", "v1_small_objects_cosine_stage_3_25_best_model.pt"),
    model_pt, device=torch.device("cuda")
)

model_pt.eval()
model_pt.transform.min_size = (1024,)
results = validate_onnx_export(
    model=model_pt,
    onnx_path=os.path.join(DATAFRAME_PATH, "checkpoints", "mask_rcnn_v1.onnx"),
    image_paths=TEST_IMAGES,
    score_thresh=0.05,
)

✓ /content/taco/batch_1/000001.jpg
✓ /content/taco/batch_3/IMG_4980.JPG
✓ /content/taco/batch_7/000021.JPG
✓ /content/taco/batch_10/000008.jpg
✗ /content/taco/batch_14/000006.jpg
    scores differ on matched: max=0.427473 mean=0.012141 | labels differ: 2/66 matched

4/5 images passed validation
max score diff : 0.000854
mean mask IoU  : 0.9985
